<a href="https://colab.research.google.com/github/gbagal400/GitTraining/blob/master/autogen_with_memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install if needed (comment if already installed)
!pip install google-generativeai nest_asyncio

In [ ]:
import os
import nest_asyncio
nest_asyncio.apply()
!pip install google-genai nest_asyncio -q
#import google.generativeai as genai
import os
from google.colab import userdata
from google import genai
# Load API key from Colab Secrets into environment variable
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [ ]:
from google import genai

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

In [ ]:
# 🔹 Memory store
memory = []

# 🔹 Gemini wrapper (NEW SDK)
def gemini_llm(prompt):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    return response.text


# 🔹 Agent class with memory awareness
class GeminiAgent:
    def __init__(self, name):
        self.name = name

    def generate_reply(self, message):
        # Include memory in prompt
        context = "\n".join(memory[-5:])  # last 5 interactions
        full_prompt = f"""
        Previous context:
        {context}

        Current task:
        {message}
        """

        print(f"\n[{self.name}] thinking...")
        response = gemini_llm(full_prompt)

        print(f"\n[{self.name}]:\n{response}")

        # Save to memory
        memory.append(f"{self.name}: {response}")

        return response


# 🔹 Create agents
planner = GeminiAgent("Planner")
assistant = GeminiAgent("Assistant")


# 🔹 Run workflow with memory
def run_agents(task):
    print("\n🟢 USER TASK:\n", task)

    memory.append(f"User: {task}")

    # Step 1: Planning
    plan = planner.generate_reply(f"Break this task into steps:\n{task}")

    # Step 2: Execution
    result = assistant.generate_reply(f"Execute this plan:\n{plan}")

    print("\n✅ FINAL OUTPUT:\n", result)

In [ ]:
# 🔹 Run
run_agents("Write a Python program to compute Fibonacci numbers up to 10")


🟢 USER TASK:
 Write a Python program to compute Fibonacci numbers up to 10

[Planner] thinking...


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 27.647598207s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-2.5-flash', 'location': 'global'},

In [ ]:
run_agents("My name is Narasimhan")
run_agents("What is my name?")


🟢 USER TASK:
 My name is Narasimhan

[Planner] thinking...


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 4.224281676s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'}, 

In [ ]:
print("\n🔹 MEMORY STATE:\n", memory)


🔹 MEMORY STATE:
 ['User: Write a Python program to compute Fibonacci numbers up to 10', 'User: Write a Python program to compute Fibonacci numbers up to 10', 'User: My name is Narasimhan']
